# CoT와 ReAct의 경계

내부 추론 전문을 요구하지 않고도 결론과 확인 가능한 근거를 받는 프롬프트를 어떻게 설계할까요?

## 패키지 준비

첫 실행에서 수업 기준 버전을 설치합니다.

In [1]:
# 현재 커널에 수업 기준 패키지 버전을 설치합니다.
%pip install -q langchain==1.4.0 langchain-openai==1.6.0 python-dotenv==1.2.3

Note: you may need to restart the kernel to use updated packages.


## API 키와 모델 준비

상위 폴더의 `.env`를 찾고 키의 존재만 확인합니다.

In [2]:
# find_dotenv는 실행 폴더부터 상위 폴더의 .env를 찾고 키 값은 출력하지 않습니다.
import json, os, time
from dotenv import find_dotenv, load_dotenv
from langchain.chat_models import init_chat_model

env_path = find_dotenv(usecwd=True)
if env_path:
    load_dotenv(env_path)
if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError("프로젝트 루트의 .env에 OPENAI_API_KEY가 필요합니다.")
model = init_chat_model("openai:gpt-5.4-mini", max_tokens=220)

## 고정 과제

모든 절에서 같은 고객 문의와 확인 단어를 사용합니다.

In [3]:
# TASK는 모든 비교에서 바꾸지 않는 합성 문의이고 MENTIONS는 품질 점수가 아닙니다.
TASK = """고객 문의: 어제 받은 텀블러 뚜껑이 깨졌습니다. 주문번호는 A-1042입니다.
새 상품으로 바꾸고 싶습니다. 회사 규정상 교환 접수에는 파손 사진이 필요합니다.
답변에는 분류, 필요한 다음 행동, 고객에게 보낼 한 문장을 포함하세요."""
MENTIONS = ["교환", "A-1042", "사진"]

## 측정 함수

단어 포함 표시는 원문 정독을 돕는 표시이며 품질 점수가 아닙니다.

In [4]:
# 모델 응답의 원문, 사용량, 경과 시간을 한 레코드로 보존합니다.
def run(messages):
    started = time.perf_counter()
    answer = model.invoke(messages)
    return {"text": answer.text, "usage": answer.usage_metadata,
            "latency_s": round(time.perf_counter() - started, 3)}

def mention_flags(text):
    return {word: word in text for word in MENTIONS}

## 짧은 최종 답 요청

세부 내부 추론은 요청하지 않고 필요한 결과만 지정합니다.

In [5]:
# 내부 추론 전문을 요구하지 않고 필요한 최종 필드만 요청합니다.
direct = run([
    ("system", "고객 지원 요청을 분류하세요."),
    ("user", "최종 답만 분류/다음 행동/고객 문장 형식으로 쓰세요.\n" + TASK),
])
print(json.dumps(direct, ensure_ascii=False, indent=2))

{
  "text": "분류: 교환 요청  \n다음 행동: 주문번호 A-1042 확인 후, 파손된 뚜껑 사진을 접수받아 교환 가능 여부를 안내해야 합니다.  \n고객 문장: 파손된 뚜껑 사진을 보내주시면 주문번호 A-1042 확인 후 새 상품으로 교환 절차를 도와드리겠습니다.",
  "usage": {
    "input_tokens": 115,
    "output_tokens": 87,
    "total_tokens": 202,
    "input_token_details": {
      "audio": 0,
      "cache_read": 0
    },
    "output_token_details": {
      "audio": 0,
      "reasoning": 0
    }
  },
  "latency_s": 1.187
}


## 확인 가능한 근거를 한 줄 추가

숨은 사고 과정을 요구하는 대신 입력에서 확인한 사실을 짧게 쓰게 합니다.

In [6]:
# 공개 가능한 짧은 근거 한 줄과 최종 답을 요청합니다.
evidence = run([
    ("system", "고객 지원 요청을 분류하세요."),
    ("user", "내부 판단 과정은 공개하지 말고, 결론과 입력에서 확인한 근거 한 줄을 쓴 뒤 "
             "분류/다음 행동/고객 문장을 제시하세요.\n" + TASK),
])
print(json.dumps(evidence, ensure_ascii=False, indent=2))

{
  "text": "- **분류:** 교환/파손 접수\n- **근거:** “어제 받은 텀블러 뚜껑이 깨졌습니다”와 “새 상품으로 바꾸고 싶습니다”라고 하셨고, “파손 사진이 필요합니다”는 접수 요건으로 확인됩니다.\n- **다음 행동:** 주문번호 **A-1042** 기준으로 파손 사진 제출을 요청한 뒤 교환 접수를 진행하세요.\n- **고객에게 보낼 한 문장:** 주문번호 A-1042 확인되었으며, 교환 접수를 위해 파손된 뚜껑 사진을 보내주시면 바로 도와드리겠습니다.",
  "usage": {
    "input_tokens": 134,
    "output_tokens": 150,
    "total_tokens": 284,
    "input_token_details": {
      "audio": 0,
      "cache_read": 0
    },
    "output_token_details": {
      "audio": 0,
      "reasoning": 0
    }
  },
  "latency_s": 1.393
}


## A/B 결과 비교

ReAct의 행동 루프는 구현하지 않습니다. 이 절에서는 텍스트 계획과 최종 답의 경계만 확인합니다.

In [7]:
# 최종 답만 요청한 조건과 짧은 근거를 추가한 조건의 길이·사용량을 비교합니다.
for name, result in {"direct": direct, "brief_evidence": evidence}.items():
    print(name, mention_flags(result["text"]), "글자 수", len(result["text"]),
          result["usage"], result["latency_s"])

direct {'교환': True, 'A-1042': True, '사진': True} 글자 수 139 {'input_tokens': 115, 'output_tokens': 87, 'total_tokens': 202, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}} 1.187
brief_evidence {'교환': True, 'A-1042': True, '사진': True} 글자 수 251 {'input_tokens': 134, 'output_tokens': 150, 'total_tokens': 284, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}} 1.393
